In [1]:
import pandas as pd

customers = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/customers_cleaned.csv")
orders = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/orders_cleaned.csv")
products = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/products_cleaned.csv")
sellers = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/sellers_cleaned.csv")
order_payments = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/order_payments_cleaned.csv")
order_reviews = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/order_reviews_cleaned.csv")
order_items = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/order_items_cleaned.csv")
product_category = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/product_category_cleaned.csv")
geolocation = pd.read_csv("C:/Users/Siddarth.b/OneDrive/Documents/DataScience_guvi/Cart2Insights_project1/data/cleaned/geolocation_cleaned.csv")

dataset = {
    "customers": customers,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "order_items": order_items,
    "product_category": product_category,
    "geolocation": geolocation  
}

In [2]:
import mysql.connector

In [3]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Memememe@07"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS cart2insights")

print("Database created successfully!")

cursor.close()
conn.close()

Database created successfully!


In [4]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Memememe@07",
    database="cart2insights"
)

cursor = conn.cursor()

print("Connected to cart2insights successfully!")

Connected to cart2insights successfully!


In [5]:
#customers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state VARCHAR(10)
)
""")

#sellers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state VARCHAR(10)
)
""")

#product_category table
cursor.execute("""
CREATE TABLE IF NOT EXISTS product_category (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
)
""")

#products table
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),
    product_name_length INT,
    product_description_length INT,
    product_photos_qty INT,
    product_weight_g FLOAT,
    product_length_cm FLOAT,
    product_height_cm FLOAT,
    product_width_cm FLOAT,
    FOREIGN KEY (product_category_name)
        REFERENCES product_category(product_category_name)
)
""")

#orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    order_status VARCHAR(30),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
)
""")

#order_items table
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price FLOAT,
    freight_value FLOAT,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),
    FOREIGN KEY (product_id)
        REFERENCES products(product_id),
    FOREIGN KEY (seller_id)
        REFERENCES sellers(seller_id)
)
""")

#order_payments table
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(30),
    payment_installments INT,
    payment_value FLOAT,
    PRIMARY KEY (order_id, payment_sequential),
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
)
""")

#order_reviews table
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_reviews (
    review_id VARCHAR(50) PRIMARY KEY,
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
)
""")

#geolocation table
cursor.execute("""
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat DECIMAL(10,7),
    geolocation_lng DECIMAL(10,7),
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(10)
)
""")

conn.commit()

print("All 9 tables created successfully!")

All 9 tables created successfully!


In [6]:
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table[0])

customers
geolocation
order_items
order_payments
order_reviews
orders
product_category
products
sellers


In [7]:
customers_data = dataset["customers"].values.tolist()
sellers_data = dataset["sellers"].values.tolist()
print("Rows ready:", len(customers_data))

print("Rows ready:", len(sellers_data))

Rows ready: 99441
Rows ready: 3095


In [ ]:
insert_query = """
INSERT IGNORE INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, customers_data)

conn.commit()

print("Customers data loaded successfully!")

Customers data loaded successfully!


In [9]:


insert_query = """
INSERT IGNORE INTO sellers (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
VALUES (%s, %s, %s, %s)
"""

cursor.executemany(insert_query, sellers_data)

conn.commit()

print("Sellers data loaded successfully!")


Sellers data loaded successfully!


In [12]:
#Product Category
product_category_data = dataset["product_category"].values.tolist()

insert_query = """
INSERT IGNORE INTO product_category (
    product_category_name,
    product_category_name_english
)
VALUES (%s, %s)
"""

cursor.executemany(insert_query, product_category_data)
conn.commit()

print("Product category loaded:", len(product_category_data))


#Products
products_data = dataset["products"].values.tolist()

insert_query = """
INSERT IGNORE INTO products (
    product_id,
    product_category_name,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, products_data)
conn.commit()

print("Products loaded:", len(products_data))


# Orders

orders_data = dataset["orders"].copy()

# Convert missing values to None for MySQL
orders_data = orders_data.where(pd.notna(orders_data), None)

orders_data = orders_data.values.tolist()

print("Rows ready:", len(orders_data))

insert_query = """
INSERT IGNORE INTO orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, orders_data)
conn.commit()

print("Orders loaded:", len(orders_data))


#Order Items
order_items_data = dataset["order_items"].values.tolist()

insert_query = """
INSERT IGNORE INTO order_items (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, order_items_data)
conn.commit()

print("Order items loaded:", len(order_items_data))


#Order Payments
order_payments_data = dataset["order_payments"].values.tolist()

insert_query = """
INSERT IGNORE INTO order_payments (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
VALUES (%s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, order_payments_data)
conn.commit()

print("Order payments loaded:", len(order_payments_data))


#Order Reviews
order_reviews_data = dataset["order_reviews"].values.tolist()

insert_query = """
INSERT IGNORE INTO order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, order_reviews_data)
conn.commit()

print("Order reviews loaded:", len(order_reviews_data))


#Geolocation
geolocation_data = dataset["geolocation"].values.tolist()

insert_query = """
INSERT IGNORE INTO geolocation (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
VALUES (%s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, geolocation_data)
conn.commit()

print("Geolocation loaded:", len(geolocation_data))


print("\nAll remaining 7 tables processed successfully!")

Product category loaded: 71
Products loaded: 32951
Rows ready: 99441
Orders loaded: 99441
Order items loaded: 112650
Order payments loaded: 103886
Order reviews loaded: 99224
Geolocation loaded: 738332

All remaining 7 tables processed successfully!


In [13]:
tables = [
    "customers",
    "sellers",
    "product_category",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "geolocation"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(table, ":", cursor.fetchone()[0])

customers : 99441
sellers : 3095
product_category : 71
products : 32328
orders : 99441
order_items : 111023
order_payments : 103886
order_reviews : 98410
geolocation : 738332
